# Global PFSS with MagWind
Run from the extracted MagWind folder after installing `python -m pip install ".[notebook]"`. The bundled HMI map works offline. To use your own data, supply one full-Sun 2D Carrington radial-field FITS map (CEA or CAR, gauss) or a regular-latitude GONG–ADAPT cube (ensemble mean). Both methods use the same area-remapped, flux-balanced photospheric boundary.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from fdips_pfss.global_field import solve

magnetogram = Path("example/hmi.synoptic_mr_polfil_720s.2219.Mr_polfil.fits")
method = "harmonic"     # "harmonic" or "fdips"
R_ss = 2.5              # Source-surface radius in solar radii
lmax = 12               # Harmonic degree limit, used only by "harmonic"
grid = (16, 48, 96)     # Radial, colatitude and longitude cell counts

In [ ]:
field = solve(magnetogram, method, R_ss, lmax, grid)
output = Path(f"{magnetogram.stem}-{method}-R{R_ss}-{'x'.join(map(str, grid))}"
              + (f"-L{lmax}" if method == "harmonic" else "") + ".npz")
np.savez_compressed(output, magnetogram=magnetogram.name, **field)
print(f"{output} | removed mean {field['removed_mean_G']:.4g} G")
if method == "fdips":
    print(f"Relative cell-flux residual {field['relative_flux_residual_L2']:.2e}")

### Inspect the source surface
`Br[r, theta, phi]` is in gauss. Colatitude increases southward and longitude increases in the Carrington direction. For harmonic PFSS, all three components use these nodes. FDIPS stores `Btheta[r_center, theta_face, phi]` and `Bphi[r_center, theta, phi_face]`. The NPZ includes these coordinates in solar radii and radians.

In [ ]:
lon = np.rad2deg(field["phi"])
lat = 90 - np.rad2deg(field["theta"])
br = field["Br"][-1]
limit = max(float(np.max(np.abs(br))), np.finfo(float).eps)
fig, ax = plt.subplots(figsize=(8, 3))
image = ax.pcolormesh(lon, lat, br, cmap="RdBu_r", vmin=-limit, vmax=limit, shading="auto")
ax.set(xlabel="Carrington longitude (deg, unwrapped)", ylabel="Latitude (deg)",
       title=f"{method} radial field at {R_ss:g} solar radii")
fig.colorbar(image, ax=ax, label="Br (G)")
plt.show()